# 03 - Champion Selection & Experiment Balance Check

This notebook:
1. Dynamically queries  for all completed LLaMEA experiments present in the database.
2. Displays an **experiment summary** grouped by problem ID, dimension, noise level, and prompt strategy.
3. Selects separate **Clean** () and **Noisy** () champion algorithms per problem (lowest  across iterations).
4. Exports  for evaluation in Notebook 05.


In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, PROJECT_ROOT
from infra.storage import get_db_connection, get_db_engine

CHAMPIONS_PATH = DATA_DIR / 'champions.json'
print(f'Champions Output Path: {CHAMPIONS_PATH}')


Champions Output Path: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/champions.json


## 1. Experiment Balance Check

In [2]:
# Query all completed experiments dynamically
query_exps = """
SELECT 
    id as exp_id,
    problem_id,
    dim,
    noise_std,
    prompt_strategy,
    llm_name,
    status,
    best_algorithm,
    best_final_error
FROM experiments
WHERE status = 'completed'
ORDER BY problem_id, dim, noise_std, prompt_strategy, exp_id
"""
with get_db_connection() as conn:
    df_exps = pd.read_sql_query(query_exps, conn)

if df_exps.empty:
    raise RuntimeError("No completed experiments found in database. Please run Notebook 02 first.")

print(f'Total completed experiments in database: {len(df_exps)}')
print('\n=== Completed Experiments Summary (Problem x Dim x Noise x Strategy) ===')
summary = df_exps.groupby(['problem_id', 'dim', 'noise_std', 'prompt_strategy']).size().reset_index(name='runs_count')
display(summary) if 'display' in globals() else print(summary.to_string(index=False))


Total completed experiments in database: 156

=== Completed Experiments Summary (Problem x Dim x Noise x Strategy) ===
 problem_id  dim  noise_std prompt_strategy  runs_count
          1    2       0.00        baseline           3
          1    2       0.00          guided           1
          1    2       0.00        thinking           1
          1    2       0.00   vectorization           1
          1    2       0.05        baseline           2
          1    2       0.05          guided           1
          1    2       0.05        thinking           1
          1    2       0.05   vectorization           1
          1    3       0.00        baseline           3
          1    3       0.00          guided           1
          1    3       0.00        thinking           1
          1    3       0.00   vectorization           1
          1    3       0.05        baseline           3
          1    3       0.05          guided           1
          1    3       0.05        thinki

## 2. Select Problem-Specific Champions (Dynamically)

In [3]:
# Query all iterations from completed experiments to find true absolute lowest final_error per problem, mode, and prompt strategy
iter_query = """
SELECT 
    e.problem_id,
    e.dim,
    e.noise_std,
    e.prompt_strategy,
    e.id as experiment_id,
    e.llm_name,
    i.id as iteration_id,
    i.algorithm_name,
    i.final_error,
    i.evaluations_used,
    i.code_path
FROM iterations i
JOIN experiments e ON i.experiment_id = e.id
WHERE e.status = 'completed'
  AND i.final_error IS NOT NULL
ORDER BY i.final_error ASC
"""
with get_db_connection() as conn:
    df_iters = pd.read_sql_query(iter_query, conn)

if df_iters.empty:
    raise RuntimeError("No completed iterations found in database.")

champions = {}

print('=== Strategy-Specific & Problem-Specific Champions ===')
for p_id in sorted(df_iters['problem_id'].unique()):
    p_subset = df_iters[df_iters['problem_id'] == p_id]
    
    for strat in sorted(p_subset['prompt_strategy'].unique()):
        strat_subset = p_subset[p_subset['prompt_strategy'] == strat]
        
        # 1. Clean Champion (noise_std == 0.0)
        clean_subset = strat_subset[strat_subset['noise_std'] == 0.0]
        if not clean_subset.empty:
            best_clean = clean_subset.iloc[0]
            key_clean = f"f{p_id}_clean_{strat}"
            champions[key_clean] = {
                'problem_id': int(p_id),
                'mode': 'clean',
                'noise_std': 0.0,
                'dim': int(best_clean['dim']),
                'prompt_strategy': str(strat),
                'experiment_id': int(best_clean['experiment_id']),
                'algorithm_name': str(best_clean['algorithm_name']),
                'final_error': float(best_clean['final_error']),
                'evaluations_used': int(best_clean['evaluations_used']) if pd.notnull(best_clean['evaluations_used']) else None,
                'code_path': str(best_clean['code_path']),
                'llm_name': str(best_clean['llm_name']),
            }
            print(f"  f{p_id} Clean  [{strat:<13}]: {best_clean['algorithm_name']} (Exp #{best_clean['experiment_id']}) -> err = {best_clean['final_error']:.6e}")
        
        # 2. Noisy Champion (noise_std > 0.0)
        noisy_subset = strat_subset[strat_subset['noise_std'] > 0.0]
        if not noisy_subset.empty:
            best_noisy = noisy_subset.iloc[0]
            key_noisy = f"f{p_id}_noisy_{strat}"
            champions[key_noisy] = {
                'problem_id': int(p_id),
                'mode': 'noisy',
                'noise_std': float(best_noisy['noise_std']),
                'dim': int(best_noisy['dim']),
                'prompt_strategy': str(strat),
                'experiment_id': int(best_noisy['experiment_id']),
                'algorithm_name': str(best_noisy['algorithm_name']),
                'final_error': float(best_noisy['final_error']),
                'evaluations_used': int(best_noisy['evaluations_used']) if pd.notnull(best_noisy['evaluations_used']) else None,
                'code_path': str(best_noisy['code_path']),
                'llm_name': str(best_noisy['llm_name']),
            }
            print(f"  f{p_id} Noisy  [{strat:<13}]: {best_noisy['algorithm_name']} (Exp #{best_noisy['experiment_id']}, std={best_noisy['noise_std']}) -> err = {best_noisy['final_error']:.6e}")

# Save champions.json
CHAMPIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(CHAMPIONS_PATH, 'w', encoding='utf-8') as f:
    json.dump(champions, f, indent=2)

print(f'\n✨ Exported {len(champions)} champion(s) to {CHAMPIONS_PATH}')


=== Strategy-Specific & Problem-Specific Champions ===
  f1 Clean  [baseline     ]: ImprovedHillClimbing (Exp #13) -> err = 0.000000e+00
  f1 Noisy  [baseline     ]: ImprovedRandomSearchOptimizer (Exp #2, std=0.05) -> err = 0.000000e+00
  f1 Clean  [guided       ]: DEOptimizer (Exp #63) -> err = 0.000000e+00
  f1 Noisy  [guided       ]: NoisyOptimizationAlgorithm (Exp #82, std=0.05) -> err = 2.330717e-02
  f1 Clean  [thinking     ]: EnhancedOptimizer (Exp #87) -> err = 0.000000e+00
  f1 Noisy  [thinking     ]: EnhancedStochasticOptimizer (Exp #142, std=0.05) -> err = 1.353932e-07
  f1 Clean  [vectorization]: EnhancedGradientDirectedSearch (Exp #59) -> err = 0.000000e+00
  f1 Noisy  [vectorization]: ImprovedNoisyOptimization (Exp #107, std=0.05) -> err = 4.944652e-03
  f8 Clean  [baseline     ]: ImprovedDifferentialEvolution (Exp #118) -> err = 0.000000e+00
  f8 Noisy  [baseline     ]: ImprovedEvolutionaryAlgorithm (Exp #16, std=0.05) -> err = 5.596235e-11
  f8 Clean  [guided       ]: E